# Stage 2 Notebook 64 - Exp2III Anchor + cls_sep + width=1.0 + full 70K + 8ep

**Push capacity from NB62's record.** NB62 (width=0.5, full 70K, cls_sep, 12 ep) hit decoded_f1=0.073, matched_iou=0.550, gap=0.045 -- NEW PROJECT RECORDS. The geometry plateaued at 0.55 with width=0.5. NB45 (width=1.0 at limit=3000) hit matched_iou=0.525, showing the capacity headroom.

Exp2III: combine NB62's recipe with width=1.0 + embed_dim=192. Hypothesis: 2x backbone capacity + 1.5x lane head capacity unblocks both geometry (matched_iou past 0.60) and cls (gap past 0.06).

Diffs vs NB62 (exp57):
- `model.width: 0.5 -> 1.0`
- `lane_head.embed_dim: 128 -> 192`
- `lane_head.roi_mid_channels: 48 -> 64`
- `end_epoch: 12 -> 8` (1.5x slower per epoch -> 8 epochs ~ NB62's 12-epoch wall clock)

GPU mem: NB60 used 10.4 GB. width=1.0 should land around 25-30 GB on the 95 GB Pro 6000.

### Run mode
1. Smoke first (new wider model).
2. 8 epochs full data. ~3-3.5 hr.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint_smoke.log
OK exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=6.3747 det_loss=3.2764 grad_cos=0.2512 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5012323260307312, 'gate/lane_mean': 0.50178462266922, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full8'
    EPOCHS = 8
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint_full8 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint_full8.tar --epochs 8 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint_full8.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp59_rmt_gca_anchor_cls_sep_vfl_w1_full_data_joint_full8_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp59_rmt_gca_anchor_cls_sep_vfl_w

0

## What to watch in Exp2III

Reference NB62 (width=0.5, cls_sep, full, 12 ep): matched_iou=0.550, decoded_f1=0.073, gap=0.045, val_lane_f1=0.118.

Pass criteria at epoch 8:
- **val/matched_line_iou >= 0.60** -- width=1.0 unblocks geometry past NB62's 0.55 plateau.
- val/lane/decoded_f1 >= 0.10 (40% over NB62).
- pos-neg gap >= 0.06.
- val/lane_f1 >= 0.15.